# PGx Cohort Feature Engineering Pipeline

This notebook orchestrates **feature engineering steps (Steps 3-5d)** of the PGx analysis pipeline with **status tracking** and **parallel execution** capabilities.

## Feature Engineering Workflow

1. **Step 3: Feature Importance** (`3_feature_importance/`)
   - Monte Carlo CV with CatBoost, XGBoost, XGBoost RF
   - Outputs: Aggregated feature importance CSVs

2. **Step 4a: Model Data Extraction** (`4a_model_data/`)
   - Creates `model_events.parquet` with cases + controls
   - Filtered by feature importance

3. **Step 4b: DTW Protocol Filtering** (`4b_dtw_filter/`)
   - Creates `model_events_no_protocols.parquet` (preferred input for downstream)

4. **Step 4c: Extreme-Density Cohort Split** (`5b_fpgrowth_analysis/extract_extreme_density_cohort.py`)
   - Extracts extreme-density patients into separate cohort
   - Removes them from main `model_events.parquet`

5. **Step 5a: BupaR Process Mining** (`5a_bupaR_analysis/`)
   - Event logs, pre/post-target features, time-to-event

6. **Step 5b: FP-Growth Analysis** (`5b_fpgrowth_analysis/`)
   - Frequent pattern mining, itemsets, association rules

7. **Step 5c: PGx Feature Engineering** (`5c_pgx_analysis/`)
   - Drug-gene mapping, allele frequencies, PGx features

8. **Step 5d: DTW Trajectory Analysis** (`5d_dtw_analysis/`)
   - Trajectory prototypes, DTW distances, patient clustering

## Notebook Features

- ✅ **Pipeline Orchestration** – Run full feature engineering pipeline or individual steps
- ✅ **Status Tracking** – Check completion status of each step per cohort/age band
- ✅ **Artifact Verification** – Automatically detect completed steps
- ✅ **Parallel Execution** – Run multiple cohorts in parallel using ProcessPoolExecutor
- ✅ **Validation Ready** – Complete feature engineering before moving to risk modeling

## Next Steps

After completing feature engineering, review and validate results, then proceed to `pgx_risk_model.ipynb` for model training and analysis.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import json
from typing import Dict, List, Tuple, Optional
from datetime import datetime

def resolve_project_root() -> Path:
    """Resolve the pgx-analysis project root for both notebook and script modes."""
    # When run as a script, __file__ is defined
    if "__file__" in globals():
        return Path(__file__).resolve().parents[0]

    # When run inside Jupyter, fall back to current working directory and parents
    cwd = Path(os.getcwd()).resolve()
    # Check for feature engineering notebook first
    if (cwd / "pgx_cohort_feature_engineering.ipynb").exists():
        return cwd
    
    # Check for original pipeline notebook
    if (cwd / "pgx_cohort_pipeline.ipynb").exists():
        return cwd

    for parent in cwd.parents:
        if (parent / "pgx_cohort_feature_engineering.ipynb").exists() or (parent / "pgx_cohort_pipeline.ipynb").exists():
            return parent

    return cwd


PROJECT_ROOT = resolve_project_root()
print(f"[INFO] Project root: {PROJECT_ROOT}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Import constants
from py_helpers.constants import COHORT_NAMES, AGE_BANDS

# ------------------------------------------------------------------
# Cohort configuration for feature engineering
# ------------------------------------------------------------------
COHORT_NAME = "opioid_ed"       # or "non_opioid_ed"
AGE_BAND = "25-44"              # e.g., "0-12", "13-24", ..., "85-94"
TRAIN_YEARS = [2016, 2017, 2018]
TEST_YEAR = 2019

print(
    f"[CONFIG] cohort={COHORT_NAME}, age_band={AGE_BAND}, "
    f"train_years={TRAIN_YEARS}, test_year={TEST_YEAR}"
)

## Feature Engineering Status Tracking

Functions to check completion status of feature engineering steps by verifying expected artifacts exist.

In [ ]:
def check_feature_engineering_status(cohort_name: str, age_band: str) -> Dict[str, str]:
    """
    Check completion status of all feature engineering steps (Steps 3-5d) for a given cohort/age band.
    
    Returns a dictionary mapping step names to status: 'DONE', 'PENDING', 'PARTIAL', or 'ERROR'
    """
    age_band_fname = age_band.replace("-", "_")
    status = {}
    
    # Step 3: Feature Importance
    fi_output = PROJECT_ROOT / "3_feature_importance" / "outputs" / cohort_name / age_band_fname
    fi_file = fi_output / f"{cohort_name}_{age_band_fname}_aggregated_feature_importance.csv"
    status["step3_feature_importance"] = "DONE" if fi_file.exists() else "PENDING"
    
    # Step 4a: Model Data
    model_data_path = (
        PROJECT_ROOT / "4a_model_data" / f"cohort_name={cohort_name}" / f"age_band={age_band}" / "model_events.parquet"
    )
    status["step4a_model_data"] = "DONE" if model_data_path.exists() else "PENDING"
    
    # Step 4b: DTW Protocol Filtering
    no_protocols_path = model_data_path.parent / "model_events_no_protocols.parquet"
    status["step4b_dtw_filter"] = "DONE" if no_protocols_path.exists() else "PENDING"
    
    # Step 4c: Extreme-Density Split
    extreme_patients_csv = model_data_path.parent / f"extreme_density_patients_{age_band_fname}.csv"
    extreme_cohort_path = (
        PROJECT_ROOT / "4a_model_data" / f"cohort_name={cohort_name}_extreme_density" / f"age_band={age_band}" / "model_events.parquet"
    )
    status["step4c_extreme_split"] = "DONE" if (extreme_patients_csv.exists() or extreme_cohort_path.exists()) else "PENDING"
    
    # Step 5a: BupaR
    bupar_features = (
        PROJECT_ROOT / "5_feature_engineering" / "feature_engineering_outputs" / "5_bupar" / cohort_name / age_band /
        f"bupaR_added_features_{cohort_name}_{age_band_fname}.csv"
    )
    # Also check alternative location
    if not bupar_features.exists():
        bupar_features = (
            PROJECT_ROOT / "5a_bupaR_analysis" / "outputs" / "feature_engineering" / 
            f"bupaR_added_features_{cohort_name}_{age_band_fname}.csv"
        )
    status["step5a_bupar"] = "DONE" if bupar_features.exists() else "PENDING"
    
    # Step 5b: FP-Growth
    fpgrowth_features = (
        PROJECT_ROOT / "5_feature_engineering" / "feature_engineering_outputs" / "4_fpgrowth" / cohort_name / age_band /
        f"fpgrowth_added_features_{cohort_name}_{age_band_fname}.csv"
    )
    # Also check alternative location
    if not fpgrowth_features.exists():
        fpgrowth_features = (
            PROJECT_ROOT / "5b_fpgrowth_analysis" / "outputs" / "feature_engineering" / 
            f"fpgrowth_added_features_{cohort_name}_{age_band_fname}.csv"
        )
    status["step5b_fpgrowth"] = "DONE" if fpgrowth_features.exists() else "PENDING"
    
    # Step 5c: PGx
    pgx_features = (
        PROJECT_ROOT / "5_feature_engineering" / "feature_engineering_outputs" / "7_pgx" / cohort_name / age_band /
        f"pgx_added_features_{cohort_name}_{age_band_fname}.csv"
    )
    # Also check alternative location
    if not pgx_features.exists():
        pgx_features = (
            PROJECT_ROOT / "5c_pgx_analysis" / "outputs" / "feature_engineering" / 
            f"pgx_added_features_{cohort_name}_{age_band_fname}.csv"
        )
    status["step5c_pgx"] = "DONE" if pgx_features.exists() else "PENDING"
    
    # Step 5d: DTW
    dtw_features = (
        PROJECT_ROOT / "5_feature_engineering" / "feature_engineering_outputs" / "6_dtw" / cohort_name / age_band /
        f"dtw_added_features_{cohort_name}_{age_band_fname}.csv"
    )
    # Also check alternative location
    if not dtw_features.exists():
        dtw_features = (
            PROJECT_ROOT / "5d_dtw_analysis" / "outputs" / "feature_engineering" / 
            f"dtw_added_features_{cohort_name}_{age_band_fname}.csv"
        )
    status["step5d_dtw"] = "DONE" if dtw_features.exists() else "PENDING"
    
    return status


def print_feature_engineering_status(cohort_name: str, age_band: str):
    """Print a formatted status table for feature engineering steps."""
    status = check_feature_engineering_status(cohort_name, age_band)
    
    print(f"\n{'='*80}")
    print(f"Feature Engineering Status: {cohort_name} / {age_band}")
    print(f"{'='*80}")
    print(f"{'Step':<30} {'Status':<15}")
    print(f"{'-'*80}")
    
    step_names = {
        "step3_feature_importance": "3. Feature Importance",
        "step4a_model_data": "4a. Model Data",
        "step4b_dtw_filter": "4b. DTW Protocol Filter",
        "step4c_extreme_split": "4c. Extreme-Density Split",
        "step5a_bupar": "5a. BupaR",
        "step5b_fpgrowth": "5b. FP-Growth",
        "step5c_pgx": "5c. PGx",
        "step5d_dtw": "5d. DTW",
    }
    
    for step_key, step_label in step_names.items():
        stat = status.get(step_key, "UNKNOWN")
        icon = "✅" if stat == "DONE" else "⏳"
        print(f"{step_label:<30} {icon} {stat:<15}")
    
    print(f"{'='*80}\n")
    
    # Summary
    done_count = sum(1 for s in status.values() if s == "DONE")
    total_count = len(status)
    print(f"Progress: {done_count}/{total_count} steps complete ({100*done_count/total_count:.1f}%)")
    
    if done_count == total_count:
        print("\n✅ Feature engineering complete! Ready to proceed to risk modeling.")
        print("   Next: Open pgx_risk_model.ipynb to train models and run analysis.")


# Check current status
print_feature_engineering_status(COHORT_NAME, AGE_BAND)

## Feature Engineering Pipeline Execution

Run individual steps or the full feature engineering pipeline. Each step checks for existing outputs and skips if already complete (idempotent).

In [ ]:
def run_feature_engineering_step(step_name: str, cohort_name: str, age_band: str, force: bool = False) -> bool:
    """
    Run a single feature engineering step.
    
    Args:
        step_name: One of '3', '4a', '4b', '4c', '5a', '5b', '5c', '5d'
        cohort_name: Cohort name (e.g., 'opioid_ed')
        age_band: Age band (e.g., '0-12')
        force: If True, rerun even if outputs exist
    
    Returns:
        True if step completed successfully, False otherwise
    """
    age_band_fname = age_band.replace("-", "_")
    python_bin = sys.executable
    
    print(f"\n{'='*80}")
    print(f"Running Step {step_name}: {cohort_name} / {age_band}")
    print(f"{'='*80}\n")
    
    try:
        if step_name == "3":
            # Feature Importance - use the cohort runner or direct script
            script = PROJECT_ROOT / "3_feature_importance" / "run_mc_feature_importance.py"
            cmd = [str(python_bin), str(script), "--cohort-name", cohort_name, "--age-band", age_band]
            
        elif step_name == "4a":
            script = PROJECT_ROOT / "4a_model_data" / "create_model_data.py"
            cmd = [str(python_bin), str(script), "--cohort-name", cohort_name, "--age-band", age_band]
            
        elif step_name == "4b":
            script = PROJECT_ROOT / "4b_dtw_filter" / "filter_protocol_events.py"
            cmd = [str(python_bin), str(script), "--cohort-name", cohort_name, "--age-band", age_band]
            
        elif step_name == "4c":
            script = PROJECT_ROOT / "5b_fpgrowth_analysis" / "extract_extreme_density_cohort.py"
            cmd = [str(python_bin), str(script), "--cohort-name", cohort_name, "--age-band", age_band]
            
        elif step_name == "5a":
            script = PROJECT_ROOT / "5a_bupaR_analysis" / "run_analysis.py"
            cmd = [str(python_bin), str(script), "--cohort-name", cohort_name, "--age-band", age_band]
            
        elif step_name == "5b":
            script = PROJECT_ROOT / "5b_fpgrowth_analysis" / "run_analysis.py"
            cmd = [str(python_bin), str(script), "--cohort-name", cohort_name, "--age-band", age_band]
            
        elif step_name == "5c":
            script = PROJECT_ROOT / "5c_pgx_analysis" / "run_analysis.py"
            cmd = [str(python_bin), str(script), "--cohort-name", cohort_name, "--age-band", age_band]
            
        elif step_name == "5d":
            # DTW has two steps
            script1 = PROJECT_ROOT / "5d_dtw_analysis" / "create_dtw_features.py"
            script2 = PROJECT_ROOT / "5d_dtw_analysis" / "add_dtw_features_to_model_data.py"
            cmd1 = [str(python_bin), str(script1), "--cohort", cohort_name, "--age_band", age_band]
            cmd2 = [str(python_bin), str(script2), "--cohort-name", cohort_name, "--age-band", age_band]
            
            print(f"Running DTW Step 1/2: create_dtw_features.py")
            result1 = subprocess.run(cmd1, cwd=PROJECT_ROOT, check=False)
            if result1.returncode != 0:
                print(f"❌ DTW Step 1 failed with return code {result1.returncode}")
                return False
            
            print(f"Running DTW Step 2/2: add_dtw_features_to_model_data.py")
            result2 = subprocess.run(cmd2, cwd=PROJECT_ROOT, check=False)
            if result2.returncode != 0:
                print(f"❌ DTW Step 2 failed with return code {result2.returncode}")
                return False
            
            print(f"✅ Step 5d completed successfully")
            return True
            
        else:
            print(f"❌ Unknown step: {step_name}")
            return False
        
        # Check if step already complete (unless force)
        if not force and step_name != "5d":  # 5d handled separately
            status = check_feature_engineering_status(cohort_name, age_band)
            step_key = f"step{step_name}"
            if step_key in status and status[step_key] == "DONE":
                print(f"⏭️  Step {step_name} already complete. Use force=True to rerun.")
                return True
        
        # Run the command
        if step_name != "5d":  # Already handled above
            result = subprocess.run(cmd, cwd=PROJECT_ROOT, check=False)
            if result.returncode == 0:
                print(f"✅ Step {step_name} completed successfully")
                return True
            else:
                print(f"❌ Step {step_name} failed with return code {result.returncode}")
                return False
                
    except Exception as e:
        print(f"❌ Error running step {step_name}: {e}")
        return False


def run_full_feature_engineering(cohort_name: str, age_band: str, start_from: Optional[str] = None, force: bool = False):
    """
    Run the full feature engineering pipeline from a given step (or from the beginning).
    
    Args:
        cohort_name: Cohort name
        age_band: Age band
        start_from: Step to start from (e.g., '4a', '5b'). If None, starts from Step 3.
        force: If True, rerun steps even if outputs exist
    """
    steps = ["3", "4a", "4b", "4c", "5b", "5a", "5c", "5d"]
    
    if start_from:
        try:
            start_idx = steps.index(start_from)
            steps = steps[start_idx:]
        except ValueError:
            print(f"❌ Unknown step: {start_from}. Available steps: {', '.join(steps)}")
            return
    
    print(f"\n{'='*80}")
    print(f"Running Full Feature Engineering Pipeline: {cohort_name} / {age_band}")
    if start_from:
        print(f"Starting from step: {start_from}")
    print(f"{'='*80}\n")
    
    for step in steps:
        success = run_feature_engineering_step(step, cohort_name, age_band, force=force)
        if not success:
            print(f"\n❌ Pipeline stopped at step {step}")
            print(f"Fix the error and rerun from step {step} using:")
            print(f"  run_full_feature_engineering('{cohort_name}', '{age_band}', start_from='{step}')")
            return
        
        # Update status after each step
        print_feature_engineering_status(cohort_name, age_band)
    
    print(f"\n{'='*80}")
    print(f"✅ Feature engineering pipeline completed for {cohort_name} / {age_band}")
    print(f"{'='*80}\n")
    print("📋 Next: Review and validate feature engineering results, then proceed to pgx_risk_model.ipynb")


# Example: Run a single step
# run_feature_engineering_step("4a", COHORT_NAME, AGE_BAND)

# Example: Run full feature engineering pipeline
# run_full_feature_engineering(COHORT_NAME, AGE_BAND)

## Batch Status Overview

Check feature engineering status across all cohorts and age bands.

In [ ]:
def get_all_cohort_combinations() -> List[Tuple[str, str]]:
    """Get all valid cohort/age band combinations."""
    combinations = []
    
    # Cohort 1: opioid_ed (age bands < 65)
    opioid_age_bands = ["0-12", "13-24", "25-44", "45-54", "55-64"]
    for ab in opioid_age_bands:
        combinations.append(("opioid_ed", ab))
    
    # Cohort 2: non_opioid_ed (age bands >= 65)
    non_opioid_age_bands = ["65-74", "75-84", "85-94"]
    for ab in non_opioid_age_bands:
        combinations.append(("non_opioid_ed", ab))
    
    return combinations


def print_batch_feature_engineering_status():
    """Print a comprehensive status table for all cohort/age band combinations."""
    combinations = get_all_cohort_combinations()
    
    print(f"\n{'='*100}")
    print(f"Batch Feature Engineering Status - All Cohorts")
    print(f"{'='*100}")
    print(f"{'Cohort':<25} {'Age Band':<12} {'3':<6} {'4a':<6} {'4b':<6} {'4c':<6} {'5a':<6} {'5b':<6} {'5c':<6} {'5d':<6}")
    print(f"{'-'*100}")
    
    step_keys = ["step3_feature_importance", "step4a_model_data", "step4b_dtw_filter", "step4c_extreme_split", 
                 "step5a_bupar", "step5b_fpgrowth", "step5c_pgx", "step5d_dtw"]
    step_labels = ["3", "4a", "4b", "4c", "5a", "5b", "5c", "5d"]
    
    for cohort_name, age_band in combinations:
        status = check_feature_engineering_status(cohort_name, age_band)
        row = f"{cohort_name:<25} {age_band:<12}"
        
        for step_key, step_label in zip(step_keys, step_labels):
            stat = status.get(step_key, "UNKNOWN")
            icon = "✅" if stat == "DONE" else "⏳"
            row += f"{icon:<6}"
        
        print(row)
    
    print(f"{'-'*100}")
    
    # Summary statistics
    total_combinations = len(combinations)
    total_steps = len(step_keys)
    total_cells = total_combinations * total_steps
    
    done_count = 0
    for cohort_name, age_band in combinations:
        status = check_feature_engineering_status(cohort_name, age_band)
        done_count += sum(1 for step_key in step_keys if status.get(step_key) == "DONE")
    
    print(f"\nSummary: {done_count}/{total_cells} step/cohort combinations complete ({100*done_count/total_cells:.1f}%)")
    print(f"{'='*100}\n")


# Print batch status
print_batch_feature_engineering_status()

## Parallel Cohort Execution

Run multiple cohorts in parallel using ProcessPoolExecutor (following the pattern from `7_update_codes.py` and parallelization guide).

**Configuration:**
- Uses `ProcessPoolExecutor` for multi-process parallelization (bypasses Python GIL)
- Worker count from environment variable `PGX_WORKERS_MEDICAL` or defaults to CPU count - 4
- Each worker runs the full feature engineering pipeline for one cohort/age band combination
- Progress tracking and error handling included

**R Script Support:**
- Step 5a (BupaR) uses Python wrapper `run_analysis.py` which internally calls R scripts via `Rscript`
- The parallel execution function calls the Python wrapper, which handles R execution automatically
- No special handling needed - R scripts are transparently executed through their Python wrappers

In [ ]:
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed
import os
import subprocess

def _run_feature_engineering_worker(
    project_root: str,
    cohort_name: str, 
    age_band: str, 
    start_from: Optional[str] = None, 
    force: bool = False
) -> Dict[str, Any]:
    """
    Worker function to run full feature engineering pipeline for a single cohort/age band.
    Uses subprocess to call pipeline scripts (following pattern from parallelization guide).
    
    Args:
        project_root: Path to project root (passed as string for pickling)
        cohort_name: Cohort name
        age_band: Age band
        start_from: Optional step to start from
        force: If True, rerun even if outputs exist
    
    Returns:
        Dictionary with results: {'cohort': str, 'age_band': str, 'success': bool, 'error': str or None}
    """
    import sys
    from pathlib import Path
    
    project_root = Path(project_root)
    python_bin = sys.executable
    
    # Define feature engineering steps in order
    steps = ["3", "4a", "4b", "4c", "5b", "5a", "5c", "5d"]
    
    # Filter steps if start_from is specified
    if start_from:
        try:
            start_idx = steps.index(start_from)
            steps = steps[start_idx:]
        except ValueError:
            return {
                'cohort': cohort_name,
                'age_band': age_band,
                'success': False,
                'error': f"Unknown step: {start_from}"
            }
    
    # Run each step sequentially for this cohort
    for step in steps:
        try:
            if step == "3":
                script = project_root / "3_feature_importance" / "run_mc_feature_importance.py"
                cmd = [str(python_bin), str(script), "--cohort-name", cohort_name, "--age-band", age_band]
            elif step == "4a":
                script = project_root / "4a_model_data" / "create_model_data.py"
                cmd = [str(python_bin), str(script), "--cohort-name", cohort_name, "--age-band", age_band]
            elif step == "4b":
                script = project_root / "4b_dtw_filter" / "filter_protocol_events.py"
                cmd = [str(python_bin), str(script), "--cohort-name", cohort_name, "--age-band", age_band]
            elif step == "4c":
                script = project_root / "5b_fpgrowth_analysis" / "extract_extreme_density_cohort.py"
                cmd = [str(python_bin), str(script), "--cohort-name", cohort_name, "--age-band", age_band]
            elif step == "5a":
                # BupaR uses Python wrapper that internally calls R scripts via Rscript
                script = project_root / "5a_bupaR_analysis" / "run_analysis.py"
                cmd = [str(python_bin), str(script), "--cohort-name", cohort_name, "--age-band", age_band]
            elif step == "5b":
                script = project_root / "5b_fpgrowth_analysis" / "run_analysis.py"
                cmd = [str(python_bin), str(script), "--cohort-name", cohort_name, "--age-band", age_band]
            elif step == "5c":
                script = project_root / "5c_pgx_analysis" / "run_analysis.py"
                cmd = [str(python_bin), str(script), "--cohort-name", cohort_name, "--age-band", age_band]
            elif step == "5d":
                # DTW has two steps
                script1 = project_root / "5d_dtw_analysis" / "create_dtw_features.py"
                script2 = project_root / "5d_dtw_analysis" / "add_dtw_features_to_model_data.py"
                cmd1 = [str(python_bin), str(script1), "--cohort", cohort_name, "--age_band", age_band]
                cmd2 = [str(python_bin), str(script2), "--cohort-name", cohort_name, "--age-band", age_band]
                
                result1 = subprocess.run(cmd1, cwd=project_root, capture_output=True, text=True)
                if result1.returncode != 0:
                    return {
                        'cohort': cohort_name,
                        'age_band': age_band,
                        'success': False,
                        'error': f"Step 5d (create_dtw_features) failed: {result1.stderr[:500]}"
                    }
                
                result2 = subprocess.run(cmd2, cwd=project_root, capture_output=True, text=True)
                if result2.returncode != 0:
                    return {
                        'cohort': cohort_name,
                        'age_band': age_band,
                        'success': False,
                        'error': f"Step 5d (add_dtw_features) failed: {result2.stderr[:500]}"
                    }
                continue  # Skip to next step
            else:
                return {
                    'cohort': cohort_name,
                    'age_band': age_band,
                    'success': False,
                    'error': f"Unknown step: {step}"
                }
            
            # Run the command
            result = subprocess.run(cmd, cwd=project_root, capture_output=True, text=True)
            
            if result.returncode != 0:
                return {
                    'cohort': cohort_name,
                    'age_band': age_band,
                    'success': False,
                    'error': f"Step {step} failed with return code {result.returncode}: {result.stderr[:500]}"
                }
                
        except Exception as e:
            return {
                'cohort': cohort_name,
                'age_band': age_band,
                'success': False,
                'error': f"Step {step} exception: {str(e)}"
            }
    
    # All steps completed successfully
    return {
        'cohort': cohort_name,
        'age_band': age_band,
        'success': True,
        'error': None
    }


def run_feature_engineering_parallel(
    cohort_combinations: List[Tuple[str, str]],
    start_from: Optional[str] = None,
    force: bool = False,
    max_workers: Optional[int] = None
) -> Dict[str, Any]:
    """
    Run feature engineering pipeline for multiple cohort/age band combinations in parallel.
    
    Args:
        cohort_combinations: List of (cohort_name, age_band) tuples
        start_from: Optional step to start from (e.g., '4a', '5b')
        force: If True, rerun even if outputs exist
        max_workers: Maximum number of parallel workers (defaults to CPU count - 4)
    
    Returns:
        Dictionary with results for each cohort
    """
    # Determine worker count (following parallelization guide pattern)
    if max_workers is None:
        # Use environment variable or default to CPU count - 4
        env_workers = os.getenv('PGX_WORKERS_MEDICAL')
        if env_workers and env_workers.isdigit() and int(env_workers) > 0:
            max_workers = int(env_workers)
        else:
            # Default: CPU count - 4 (leave cores for system/OS)
            max_workers = max(1, multiprocessing.cpu_count() - 4)
    
    print(f"\n{'='*80}")
    print(f"Running {len(cohort_combinations)} cohorts in parallel with {max_workers} workers")
    print(f"{'='*80}\n")
    
    results = {}
    completed = 0
    total = len(cohort_combinations)
    
    # Use ProcessPoolExecutor (following pattern from 7_update_codes.py and parallelization guide)
    # Pass project_root as string for pickling across processes
    project_root_str = str(PROJECT_ROOT)
    
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        # Submit all jobs (pass project_root_str for worker function)
        future_to_cohort = {
            executor.submit(_run_feature_engineering_worker, project_root_str, cohort, age_band, start_from, force): (cohort, age_band)
            for cohort, age_band in cohort_combinations
        }
        
        print(f"Submitted {len(future_to_cohort)} jobs to {max_workers} workers\n")
        
        # Process completed jobs as they finish
        for future in as_completed(future_to_cohort):
            cohort, age_band = future_to_cohort[future]
            completed += 1
            
            try:
                result = future.result()
                results[f"{cohort}_{age_band}"] = result
                
                if result['success']:
                    print(f"✅ [{completed}/{total}] {cohort} / {age_band} - Completed successfully")
                else:
                    print(f"❌ [{completed}/{total}] {cohort} / {age_band} - Failed: {result.get('error', 'Unknown error')}")
                    
            except Exception as e:
                results[f"{cohort}_{age_band}"] = {
                    'cohort': cohort,
                    'age_band': age_band,
                    'success': False,
                    'error': str(e)
                }
                print(f"❌ [{completed}/{total}] {cohort} / {age_band} - Exception: {e}")
    
    # Summary
    print(f"\n{'='*80}")
    print(f"Parallel Execution Summary")
    print(f"{'='*80}")
    successful = sum(1 for r in results.values() if r.get('success', False))
    failed = len(results) - successful
    print(f"Total: {len(results)} cohorts")
    print(f"✅ Successful: {successful}")
    print(f"❌ Failed: {failed}")
    print(f"{'='*80}\n")
    
    if successful == len(results):
        print("✅ All feature engineering complete! Ready to proceed to risk modeling.")
        print("   Next: Review and validate results, then open pgx_risk_model.ipynb")
    
    return results


# Example: Run all pending cohorts in parallel (optimal for EC2: 2 cohorts at a time)
# pending_cohorts = [
#     ("opioid_ed", "0-12"),
#     ("opioid_ed", "55-64"),
#     ("non_opioid_ed", "65-74"),
#     ("non_opioid_ed", "75-84"),
#     ("non_opioid_ed", "85-94"),
# ]
# results = run_feature_engineering_parallel(pending_cohorts, start_from="4b", max_workers=2)